<a href="https://colab.research.google.com/github/hwanginseo04/-/blob/main/4%EC%9B%943%EC%9D%BC%EA%B3%BC%EC%A0%9C.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

In [ ]:
url = "https://books.toscrape.com/catalogue/page-1.html"
response = requests.get(url)
soup = BeautifulSoup(response.text, 'html.parser')

In [ ]:
titles = []
books = soup.find_all('h3')

for book in books:
    title = book.find('a')['title']  # <a> 태그의 title 속성값 추출
    titles.append(title)

In [ ]:
df_titles_20 = pd.DataFrame(titles[:20], columns=['Book Title'])
print("--- 상위 20개 책 제목 ---")
print(df_titles_20)

--- 상위 20개 책 제목 ---
                                           Book Title
0                                A Light in the Attic
1                                  Tipping the Velvet
2                                          Soumission
3                                       Sharp Objects
4               Sapiens: A Brief History of Humankind
5                                     The Requiem Red
6   The Dirty Little Secrets of Getting Your Dream...
7   The Coming Woman: A Novel Based on the Life of...
8   The Boys in the Boat: Nine Americans and Their...
9                                     The Black Maria
10     Starving Hearts (Triangular Trade Trilogy, #1)
11                              Shakespeare's Sonnets
12                                        Set Me Free
13  Scott Pilgrim's Precious Little Life (Scott Pi...
14                          Rip it Up and Start Again
15  Our Band Could Be Your Life: Scenes from the A...
16                                               Olio
17  Mesa

In [ ]:
all_titles_100 = []

for page in range(1, 6):  # 1페이지부터 5페이지까지
    url = f"https://books.toscrape.com/catalogue/page-{page}.html"
    res = requests.get(url)
    soup = BeautifulSoup(res.text, 'html.parser')

    books = soup.find_all('h3')
    for book in books:
        all_titles_100.append(book.find('a')['title'])

# 데이터프레임 생성
df_titles_100 = pd.DataFrame(all_titles_100, columns=['Book Title'])
print(f"\n총 수집된 제목 수: {len(df_titles_100)}")


총 수집된 제목 수: 100


In [ ]:
import re

categories = [
    'travel_2', 'mystery_3', 'historical-fiction_4', 'sequential-art_5',
    'classics_6', 'philosophy_7', 'romance_8', 'womens-fiction_9',
    'fiction_10', 'childrens_11'
]

base_url = "https://books.toscrape.com/catalogue/category/books/"
all_data = []

for cat in categories:
    url = f"{base_url}{cat}/index.html"
    res = requests.get(url)
    soup = BeautifulSoup(res.text, 'html.parser')

    # 각 카테고리에서 책 리스트 가져오기 (최대 10개)
    items = soup.find_all('article', class_='product_pod')[:10]

    for item in items:
        title = item.find('h3').find('a')['title']
        # 가격 문자열에서 숫자만 추출 (예: £51.77 -> 51.77)
        price_text = item.find('p', class_='price_color').text
        price = float(re.sub(r'[^\d.]', '', price_text))

        all_data.append({'Category': cat.split('_')[0].replace('-', ' ').title(),
                         'Title': title,
                         'Price': price})

# 1. 데이터프레임 변환
df_final = pd.DataFrame(all_data)

# 2. 가격 기준 내림차순 정렬 후 상위 10개 추출
top_10_expensive = df_final.sort_values(by='Price', ascending=False).head(10)

print("\n--- 가장 비싼 책 Top 10 ---")
print(top_10_expensive[['Category', 'Title', 'Price']])


--- 가장 비싼 책 Top 10 ---
              Category                                              Title  \
46            Classics                                            Candide   
51          Philosophy       The Death of Humanity: and the Case for Life   
93           Childrens  The White Cat and the Monk: A Retelling of the...   
70      Womens Fiction  I Had a Nice Time And Other Lies...: How to fi...   
47            Classics                                        Animal Farm   
7               Travel                   A Year in Provence (Provence #1)   
12             Mystery                                The Past Never Ends   
92           Childrens                    The Secret of Dreadwillow Carse   
66             Romance                   Suddenly in Love (Lake Haven #1)   
22  Historical Fiction            A Flight of Arrows (The Pathfinders #2)   

    Price  
46  58.63  
51  58.11  
93  58.08  
70  57.36  
47  57.22  
7   56.88  
12  56.50  
92  56.13  
66  55.99  
22  55.5

In [ ]:
from scipy import stats

# 1. 샤피로-윌크 검정 (Shapiro-Wilk Test) 실행
# p-value가 0.05보다 크면 정규분포로 간주, 작으면 정규분포가 아님
shapiro_test = stats.shapiro(df_final['Price'])

# 2. 왜도(Skewness)와 첨도(Kurtosis) 계산
# 왜도: 0에 가까울수록 대칭 / 첨도: 0에 가까울수록 정규분포의 높이와 비슷
skewness = df_final['Price'].skew()
kurtosis = df_final['Price'].kurtosis()

print("--- 통계적 유의미함 분석 결과 ---")
print(f"Shapiro-Wilk 통계량: {shapiro_test.statistic:.4f}")
print(f"p-value: {shapiro_test.pvalue:.4f}")
print(f"왜도 (Skewness): {skewness:.4f}")
print(f"첨도 (Kurtosis): {kurtosis:.4f}")

# 3. 해석 출력
if shapiro_test.pvalue > 0.05:
    print("\n결과: p-value가 0.05보다 크므로, 귀무가설을 채택하여 '정규분포를 따른다'고 볼 수 있습니다.")
else:
    print("\n결과: p-value가 0.05보다 작으므로, 귀무가설을 기각하여 '정규분포를 따르지 않는다'고 판단합니다.")

--- 통계적 유의미함 분석 결과 ---
Shapiro-Wilk 통계량: 0.9158
p-value: 0.0000
왜도 (Skewness): 0.0545
첨도 (Kurtosis): -1.4777

결과: p-value가 0.05보다 작으므로, 귀무가설을 기각하여 '정규분포를 따르지 않는다'고 판단합니다.
